# Chapter 17: Balanced Trees: AVL

The programs from Chapter 17 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## The Balance Rule

`avl.py`

In [1]:
"""An AVL tree: a binary search tree that rebalances itself.

Every node records its height. After each insert or delete, any node
whose two subtrees differ in height by more than one is fixed with one
or two rotations, so the tree's height stays below 1.44 log2(n + 2).
"""


class Node:
    __slots__ = ("key", "value", "left", "right", "height")

    def __init__(self, key, value):
        self.key, self.value = key, value
        self.left = self.right = None
        self.height = 0


def height(node):
    return node.height if node else -1


def update(node):
    node.height = 1 + max(height(node.left), height(node.right))


def balance(node):
    """Positive when the left side is taller, negative when the right is."""
    return height(node.left) - height(node.right)


def rotate_right(node):
    """Lift the left child above node; returns the new subtree root.

          node            left
          /  \\           /  \\
        left  c   ->     a   node
        /  \\                 /  \\
       a    b               b    c
    """
    left = node.left
    node.left, left.right = left.right, node
    update(node)
    update(left)
    return left


def rotate_left(node):
    """The mirror image of rotate_right."""
    right = node.right
    node.right, right.left = right.left, node
    update(node)
    update(right)
    return right


def rebalance(node):
    update(node)
    if balance(node) > 1:  # left side too tall
        if balance(node.left) < 0:  # left-right case: two rotations
            node.left = rotate_left(node.left)
        return rotate_right(node)
    if balance(node) < -1:  # right side too tall
        if balance(node.right) > 0:  # right-left case
            node.right = rotate_right(node.right)
        return rotate_left(node)
    return node


class AVLTree:
    def __init__(self):
        self.root = None
        self._n = 0

    def __len__(self):
        return self._n

    def get(self, key, default=None):
        node = self.root
        while node:
            if key == node.key:
                return node.value
            node = node.left if key < node.key else node.right
        return default

    def __contains__(self, key):
        node = self.root
        while node and node.key != key:
            node = node.left if key < node.key else node.right
        return node is not None

    def put(self, key, value):
        self.root = self._put(self.root, key, value)

    def _put(self, node, key, value):
        if node is None:
            self._n += 1
            return Node(key, value)
        if key == node.key:
            node.value = value
            return node
        if key < node.key:
            node.left = self._put(node.left, key, value)
        else:
            node.right = self._put(node.right, key, value)
        return rebalance(node)  # fix every node on the way back up

    def delete(self, key):
        self.root = self._delete(self.root, key)
        self._n -= 1

    def _delete(self, node, key):
        if node is None:
            raise KeyError(key)
        if key < node.key:
            node.left = self._delete(node.left, key)
        elif key > node.key:
            node.right = self._delete(node.right, key)
        elif node.left and node.right:
            successor = node.right  # the next key takes this node's place
            while successor.left:
                successor = successor.left
            node.key, node.value = successor.key, successor.value
            node.right = self._delete(node.right, successor.key)
        else:
            return node.left or node.right
        return rebalance(node)

    def __iter__(self):
        stack, node = [], self.root
        while stack or node:
            while node:
                stack.append(node)
                node = node.left
            node = stack.pop()
            yield node.key
            node = node.right

    def height(self):
        return height(self.root)

## Inserting and Deleting

`rotation_demo.py`

In [2]:
"""Insert 1 to 7 in order. A plain BST would become a chain; the AVL
tree rotates and ends perfectly balanced."""

from avl import AVLTree


def draw(node, depth=0):
    if node:
        draw(node.right, depth + 1)
        print("    " * depth + str(node.key))
        draw(node.left, depth + 1)


t = AVLTree()
for k in range(1, 8):
    t.put(k, None)
    if k in (3, 5, 7):
        print(f"after inserting 1 to {k}:")
        draw(t.root)

after inserting 1 to 3:
    3
2
    1
after inserting 1 to 5:
        5
    4
        3
2
    1
after inserting 1 to 7:
        7
    6
        5
4
        3
    2
        1


## How Tall?

`heights.py`

In [3]:
"""Heights after inserting the keys 0 to n - 1 in sorted order."""

import math

from avl import AVLTree

print(
    f"{'n':>8} {'plain BST':>10} {'AVL':>4} {'log2 n':>7} "
    f"{'1.44 log2 n':>12}"
)
for n in [100, 1_000, 10_000, 100_000]:
    t = AVLTree()
    for k in range(n):
        t.put(k, None)
    print(
        f"{n:>8,} {n - 1:>10,} {t.height():>4} {math.log2(n):>7.1f} "
        f"{1.44 * math.log2(n):>12.1f}"
    )

       n  plain BST  AVL  log2 n  1.44 log2 n
     100         99    6     6.6          9.6
   1,000        999    9    10.0         14.4
  10,000      9,999   13    13.3         19.1


 100,000     99,999   16    16.6         23.9


## Timing

`bench_avl.py`

In [4]:
"""Build a tree from n sorted keys, then look up every key: a plain
BST against an AVL tree."""

import bench
from avl import AVLTree
from bst import BST  # a copy of Chapter 16's bst.py


def build_and_search(make, n):
    t = make()
    for k in range(n):
        t.put(k, None)
    for k in range(n):
        k in t


sizes = [500, 1_000, 2_000, 4_000]
results = {
    "BST": bench.doubling(build_and_search, lambda n: (BST, n), sizes),
    "AVLTree": bench.doubling(
        build_and_search, lambda n: (AVLTree, n), sizes
    ),
}
bench.table(results)
bench.chart(results, "ch17_avl.png", "Sorted keys: build, then search")

         n             BST         AVLTree
       500         7.04 ms         1.56 ms
     1,000    27.9 ms x4.0    3.56 ms x2.3
     2,000     113 ms x4.0    7.62 ms x2.1
     4,000     447 ms x4.0    16.9 ms x2.2
